## Bank of Abyssinia Review Analysis

Import Required Libraries

In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath('C:/Users/dagic/OneDrive/Documents/KAIM/Week_2/fintech-review-analytics'))

print('Path set. Python will now look in:', os.path.abspath('C:/Users/dagic/OneDrive/Documents/KAIM/Week_2/fintech-review-analytics'))

Path set. Python will now look in: C:\Users\dagic\OneDrive\Documents\KAIM\Week_2\fintech-review-analytics


In [2]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime
from src.data_scrapper import scrape_metadata, scrape_reviews, data_quality_check
from src.data_preprocessor import missing_values, duplicate_reviews, check_dateFormat, missing_values, normalize_date, clean_text, invalid_reviews, remove_duplicates, save_cleaned_data, preprocessing_report

# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

Libraries loaded successfully!


Defining Variables

In [3]:
App_Id = "com.boa.boaMobileBanking"
bank = 'Bank of Abyssinia'
source = 'Google Play Store'
print(f" Scraping Review for Bank: {bank} with App ID: {App_Id} from Source: {source}")


 Scraping Review for Bank: Bank of Abyssinia with App ID: com.boa.boaMobileBanking from Source: Google Play Store


## Mobile App Review Data Scraping from Google Play Store

In [4]:
# Get meta data for the app
scrape_metadata(App_Id, bank)

App Info for Bank of Abyssinia
App Title   : BoA Mobile
Current Score: 4.3903193
Total Ratings: 9,232
Total Reviews: 1,463
Installs     : 1,000,000+


{'title': 'BoA Mobile',
 'description': 'BoA Mobile an innovative app from Bank of Abyssinia, empowers customers to effortlessly conduct a wide range of banking transactions right at their fingertips. Experience seamless and convenient banking on the go with BoA Mobile',
 'descriptionHTML': 'BoA Mobile an innovative app from Bank of Abyssinia, empowers customers to effortlessly conduct a wide range of banking transactions right at their fingertips. Experience seamless and convenient banking on the go with BoA Mobile',
 'summary': 'Mobile Banking Application',
 'installs': '1,000,000+',
 'minInstalls': 1000000,
 'realInstalls': 2386678,
 'score': 4.3903193,
 'ratings': 9232,
 'reviews': 1463,
 'histogram': [808, 294, 399, 713, 7016],
 'price': 0,
 'free': True,
 'currency': 'USD',
 'sale': False,
 'saleTime': None,
 'originalPrice': None,
 'saleText': None,
 'offersIAP': False,
 'inAppProductPrice': None,
 'developer': 'Bank of Abyssinia',
 'developerId': '7216950951017247612',
 'develo

In [5]:
#scrape 500 reviews for the app
df_reviews, continuation_token = scrape_reviews(App_Id, bank, count=500)
df_reviews = pd.DataFrame(df_reviews)

Collected 500 raw reviews for Bank of Abyssinia app.


In [6]:
print(type(df_reviews))

<class 'pandas.DataFrame'>


In [7]:
print(df_reviews.head())
print(df_reviews.columns)

                               reviewId          userName  \
0  c5eb7589-59b7-4d72-8aa9-100a703ecaa3  shambel mulugeta   
1  f2549d78-eab0-422e-aa53-16ebe68ac8e8       Jamale nuru   
2  85d9879e-2542-46aa-94b8-ffc577e19a59     Jamale n Nuru   
3  c3bb042c-844b-4580-98b9-df418622b2fb     Hamid Abdella   
4  400ce769-3726-43b2-ac4d-755b3a15f026       Eyasu Dawit   

                                           userImage  \
0  https://play-lh.googleusercontent.com/a-/ALV-U...   
1  https://play-lh.googleusercontent.com/a/ACg8oc...   
2  https://play-lh.googleusercontent.com/a/ACg8oc...   
3  https://play-lh.googleusercontent.com/a/ACg8oc...   
4  https://play-lh.googleusercontent.com/a/ACg8oc...   

                                             content  score  thumbsUpCount  \
0                                               good      5              0   
1                                             jamale      5              0   
2                                        jamale Nuru      5   

In [8]:
# Inspect what a single raw review looks like

print("Keys in a single review:")
print(list(df_reviews.iloc[0].keys()))

print("\nFirst raw review (sample):")
for key, value in df_reviews.iloc[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: c5eb7589-59b7-4d72-8aa9-100a703ecaa3
  userName: shambel mulugeta
  userImage: https://play-lh.googleusercontent.com/a-/ALV-UjX8beF1ZB4mSxvQxtq1SO5tLcK2wLGkcrIv-0qpneo4gh3pw54
  content: good
  score: 5
  thumbsUpCount: 0
  reviewCreatedVersion: 26.03.13
  at: 2026-05-14 14:18:44
  replyContent: None
  repliedAt: None
  appVersion: 26.03.13


In [9]:
# Step 3: Extract only the columns we need
raw_data = []

for r in df_reviews.itertuples():
    raw_data.append({
        'review_id': r.reviewId,
        'review'   : r.content,
        'rating'   : r.score,
        'date'     : r.at,
        'bank'     : 'Awash Bank',
        'source'   : 'Google Play'
    })

# Build a DataFrame
df_raw = pd.DataFrame(raw_data)

print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (500, 6)


,review_id,review,rating,date,bank,source
0,c5eb7589-59b7-4d72-8aa9-100a703ecaa3,good,5,2026-05-14 14:18:44,Awash Bank,Google Play
1,f2549d78-eab0-422e-aa53-16ebe68ac8e8,jamale,5,2026-05-14 11:48:21,Awash Bank,Google Play
2,85d9879e-2542-46aa-94b8-ffc577e19a59,jamale Nuru,5,2026-05-14 11:38:17,Awash Bank,Google Play
3,c3bb042c-844b-4580-98b9-df418622b2fb,it's very good app,5,2026-05-12 04:50:32,Awash Bank,Google Play
4,400ce769-3726-43b2-ac4d-755b3a15f026,this app is good but the speed of app is very ...,2,2026-05-11 11:18:54,Awash Bank,Google Play


## Exploring the Raw Data

In [10]:
# Basic shape and types
print(f"Total reviews collected: {len(df_raw)}")
print(f"\nColumn dtypes:")
print(df_raw.dtypes)

Total reviews collected: 500

Column dtypes:
review_id               str
review                  str
rating                int64
date         datetime64[us]
bank                    str
source                  str
dtype: object


In [11]:
print("Rating Distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 4) 
    print(f"{rating} stars: {bar} ({count} reviews)")


Rating Distribution:
5 stars: ██████████████████████████████████████████████████████████████████████ (281 reviews)
4 stars: █████████ (37 reviews)
3 stars: ████ (18 reviews)
2 stars: ████ (16 reviews)
1 stars: █████████████████████████████████████ (148 reviews)


In [12]:
# What does the date column look like right now?
print("Sample date values (raw):")
print(df_raw['date'].head(10).to_string())

print(f"\nDate dtype: {df_raw['date'].dtype}")

Sample date values (raw):
0   2026-05-14 14:18:44
1   2026-05-14 11:48:21
2   2026-05-14 11:38:17
3   2026-05-12 04:50:32
4   2026-05-11 11:18:54
5   2026-05-09 07:41:50
6   2026-05-08 06:47:07
7   2026-05-07 03:33:06
8   2026-05-05 04:03:08
9   2026-05-04 07:01:17

Date dtype: datetime64[us]


## Data Quality Audit

In [13]:
data_quality_check(df_raw)

Data Quality Check:
------------------------------
Total reviews collected: 500
Missing values per column:
review_id    0
review       0
rating       0
date         0
bank         0
source       0
dtype: int64


In [14]:
missing_values(df_raw)

Removed 0 rows with missing critical data
Remaining: 500 reviews


,review_id,review,rating,date,bank,source
0,c5eb7589-59b7-4d72-8aa9-100a703ecaa3,good,5,2026-05-14 14:18:44,Awash Bank,Google Play
1,f2549d78-eab0-422e-aa53-16ebe68ac8e8,jamale,5,2026-05-14 11:48:21,Awash Bank,Google Play
2,85d9879e-2542-46aa-94b8-ffc577e19a59,jamale Nuru,5,2026-05-14 11:38:17,Awash Bank,Google Play
3,c3bb042c-844b-4580-98b9-df418622b2fb,it's very good app,5,2026-05-12 04:50:32,Awash Bank,Google Play
4,400ce769-3726-43b2-ac4d-755b3a15f026,this app is good but the speed of app is very ...,2,2026-05-11 11:18:54,Awash Bank,Google Play
...,...,...,...,...,...,...
495,d5e46c35-6bbf-4ce7-b184-cdc4c1e0eab1,Verry Amazing App from all IB,5,2025-03-01 03:35:08,Awash Bank,Google Play
496,99172a2f-926c-48be-a1b0-3971b984b6b2,Not working on this days,1,2025-02-26 05:29:28,Awash Bank,Google Play
497,781ff61a-6f89-4bb1-a455-9b66400b0cd7,Thank you BoA,5,2025-02-23 03:06:47,Awash Bank,Google Play
498,cf026bee-94e7-4b9c-8a46-034354f9d171,best banking app in the wworld,5,2025-02-22 11:58:17,Awash Bank,Google Play


Check for Duplicates

In [15]:
duplicate_reviews(df_raw)

Duplicate reviews:
------------------------------
Total duplicate review IDs: 0
Total duplicate review texts: 116
Total empty reviews: 0
Total duplicate reviews: 0


In [16]:
# Copying the raw DataFrame to work on a clean version
df = df_raw.copy()
print("Data copied for preprocessing.")

Data copied for preprocessing.


Remove Missing Data

In [17]:
df = missing_values(df)

Removed 0 rows with missing critical data
Remaining: 500 reviews


Remove Duplicates

In [18]:
df = remove_duplicates(df)

Removed 0 duplicate reviews based on review_id
Remaining: 500 reviews


Normalize Date

In [19]:
check_dateFormat(df)
df = normalize_date(df)


Checking date format:
------------------------------
Sample dates: 2026-05-14 14:18:44
Data type of 'date' column: datetime64[us]
  Target format: YYYY-MM-DD (string or date object)
Dates normalized to YYYY-MM-DD format
dtype: str

Date range: 2025-02-21 to 2026-05-14


Clean white spaces

In [20]:
df = clean_text(df)

Remove Invalid Reviews

In [21]:
df = invalid_reviews(df)

Invalid ratings (outside 1–5): 0
Remaining reviews after removing invalid ratings: 500
Data type of 'rating' column: int64


## Cleaned Data

In [22]:
# Select only the 5 required columns in the right order
df_clean = df[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

TypeError: 'NoneType' object is not subscriptable

Save Cleaned Data

In [ ]:
df = save_cleaned_data(df, "cbe_reviews")

## Report for Data Preprocessing

In [ ]:
preprocessing_report(df_raw, df)